# Experiment 7: Dimensionality Reduction and Model Evaluation (With and Without PCA)

```
experiment7_pca_model_comparison.py
======================================
ICS1512 - Machine Learning Algorithms Laboratory
Experiment 6/7 (Lab Manual title: "Dimensionality Reduction and Model
Evaluation (With and Without PCA)")

Uses the reusable module ml_lab_utils.py (from Experiment 1) for:
    - EDA                         -> generate_eda_summary()
    - Classification metrics      -> classification_performance_metrics()
    - Global plot style           -> set_plot_style()

Dataset: Wisconsin Diagnostic Breast Cancer (WDBC), 569 samples, 30 numeric
features, binary target (Malignant / Benign). Loaded from a CSV file
(breast_cancer_data.csv) in the standard Kaggle WDBC format, identical to
Experiments 5 and 6. Chosen here because its 30 continuous, correlated
features make PCA-based dimensionality reduction genuinely meaningful.

10 classifiers are each trained and tuned (small grid, 5-fold CV) TWICE:
once on the original 30-feature standardized space (No-PCA) and once on a
PCA-reduced feature space (With-PCA), so every model's own hyperparameters
are re-selected in each setting rather than reused across settings.
```

## Reusable utilities (`ml_lab_utils`, from Experiment 1)

Inlined here so this notebook runs on its own without a separate `ml_lab_utils.py`.

In [ ]:
"""
ml_lab_utils.py
================
ICS1512 - Machine Learning Algorithms Laboratory
Reusable utility module used across ALL experiments.

Implements (per lab manual, Section 4):
    1. One reusable EDA function            -> generate_eda_summary()
    2. One reusable Regression function      -> train_evaluate_regression()
    3. One reusable Classification function  -> train_evaluate_classification()
    4. One reusable Regression metrics fn    -> regression_performance_metrics()
    5. One reusable Classification metrics   -> classification_performance_metrics()

Formatting rules enforced everywhere (per lab manual, Section 1):
    - Times New Roman, 15 pt for all text / legends
    - Bold, Times New Roman, 15 pt axis labels
    - Figures exported as .eps at 600 DPI (Section 3)

NOTE on fonts: "Times New Roman" itself is a proprietary Microsoft font and is
not installable on Linux. Liberation Serif is metrically-compatible (identical
glyph widths/kerning) and is registered here under the family name
"Times New Roman" so that rcParams['font.family'] = 'Times New Roman' works
transparently. On Windows/macOS, if the real Times New Roman is installed,
matplotlib will simply use that instead.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# 1. GLOBAL PLOT STYLE  (Section 1 of the manual)
# --------------------------------------------------------------------------
def set_plot_style(font_size=15):
    """
    Applies the mandatory lab formatting to every matplotlib figure:
        - Times New Roman (or metric-compatible Liberation Serif) font
        - 15 pt base font size
        - 15 pt Times New Roman legends
        - Bold, 15 pt, Times New Roman axis labels
    Call this once at the start of a notebook / script.
    """
    # Register Liberation Serif under the alias "Times New Roman" if the
    # genuine font is not present on this machine.
    installed_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in installed_fonts:
        liberation_paths = [
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf",
        ]
        for p in liberation_paths:
            if os.path.exists(p):
                fm.fontManager.addfont(p)
                # Force the registered family name to "Times New Roman"
                # (FontEntry is a frozen dataclass in modern matplotlib, so we
                # replace the last-added entry rather than mutate it in place)
                last = fm.fontManager.ttflist[-1]
                fm.fontManager.ttflist[-1] = fm.FontEntry(
                    fname=last.fname, name="Times New Roman",
                    style=last.style, variant=last.variant,
                    weight=last.weight, stretch=last.stretch, size=last.size,
                )

    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "axes.labelsize": font_size,
        "axes.labelweight": "bold",
        "axes.titlesize": font_size,
        "axes.titleweight": "bold",
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
        "figure.titlesize": font_size + 2,
        "savefig.dpi": 600,
        "figure.dpi": 150,   # screen preview; export always forced to 600 (see save)
        "svg.fonttype": "none",
    })


def _bold_axis_labels(ax, xlabel=None, ylabel=None, title=None, fs=15):
    """Helper: apply Times New Roman / Bold / 15pt to a single axis explicitly."""
    fp_bold = fm.FontProperties(family="Times New Roman", weight="bold", size=fs)
    fp_reg = fm.FontProperties(family="Times New Roman", size=fs - 2)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=fp_bold)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=fp_bold)
    if title is not None:
        ax.set_title(title, fontproperties=fp_bold, fontsize=fs)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(fp_reg)
    leg = ax.get_legend()
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontproperties(fp_reg)


def _save_eps(fig, save_path, also_png=True):
    """Export a figure as .eps at 600 DPI (Section 3 of the manual).

    If also_png is True, an additional .png copy is saved alongside the .eps
    (same basename) purely so the figure can be embedded when compiling the
    LaTeX report with pdflatex/xelatex, which cannot rasterize .eps directly
    without Ghostscript. The .eps remains the official, mandated deliverable.
    """
    if save_path is None:
        return None
    if not save_path.lower().endswith(".eps"):
        save_path = os.path.splitext(save_path)[0] + ".eps"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, format="eps", dpi=600, bbox_inches="tight")
    if also_png:
        png_path = os.path.splitext(save_path)[0] + ".png"
        fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
    return save_path


# --------------------------------------------------------------------------
# 2. GENERIC EDA FUNCTION  (Section 4.1)  -> ONE consolidated 12-subplot figure
# --------------------------------------------------------------------------
def generate_eda_summary(df, target_col=None, dataset_name="Dataset",
                          save_path=None, figsize=(22, 16)):
    """
    Generic, reusable EDA function that works on ANY tabular dataset
    (classification, regression, or unlabeled). Produces ONE consolidated
    figure containing 12 EDA subplots on a single page, per Section 2 of the
    lab manual.

    Parameters
    ----------
    df : pandas.DataFrame
        The full dataset (features + target, if any).
    target_col : str or None
        Name of the target/label column, if present. If None, the function
        treats the dataset as unlabeled and adapts the 12-panel layout
        accordingly (no class-distribution / target-correlation panels).
    dataset_name : str
        Used in the figure's suptitle.
    save_path : str or None
        If given, the figure is exported as .eps @ 600 DPI to this path.
    figsize : tuple
        Overall figure size in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    set_plot_style()
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    is_classification_target = (
        target_col is not None and
        (df[target_col].dtype == "object" or df[target_col].nunique() <= 20)
    )

    # Pick the most "informative" numeric feature (highest variance) as the
    # representative single feature for panels 6/8/9/10, instead of blindly
    # using the first column (which can be degenerate/constant, e.g. corner
    # pixels in an image dataset such as MNIST/Digits).
    if numeric_cols:
        # Prefer genuinely continuous columns (more than 5 distinct values) so
        # binary/near-constant encoded columns (e.g. a 0/1 "sex" flag, or
        # constant corner pixels in image data) are not picked as the
        # representative single feature for panels 6/8/9/10.
        continuous_cols = [c for c in numeric_cols if df[c].nunique() > 5]
        candidate_cols = continuous_cols if continuous_cols else numeric_cols
        variances = df[candidate_cols].var().sort_values(ascending=False)
        top_var_cols = variances.index.tolist()
        feat_a = top_var_cols[0]
        feat_b = top_var_cols[1] if len(top_var_cols) > 1 else top_var_cols[0]
        kde_cols = top_var_cols[:4]
    else:
        feat_a = feat_b = None
        kde_cols = []

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"Exploratory Data Analysis Summary \u2013 {dataset_name}",
                 fontweight="bold", fontsize=17,
                 fontproperties=fm.FontProperties(family="Times New Roman",
                                                   weight="bold", size=17))
    gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)
    axes = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(12)]
    panel = 0

    # ---- Panel 1: Dataset overview (head / shape as a text table) ----
    ax = axes[panel]; panel += 1
    ax.axis("off")
    overview_txt = (
        f"Shape: {df.shape[0]} rows x {df.shape[1]} cols\n"
        f"Numeric features: {len(numeric_cols)}\n"
        f"Categorical features: {len(categorical_cols)}\n"
        f"Missing cells: {int(df.isnull().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}"
    )
    ax.text(0.02, 0.9, overview_txt, va="top", ha="left",
            fontproperties=fm.FontProperties(family="Times New Roman", size=13),
            transform=ax.transAxes)
    _bold_axis_labels(ax, title="1. Dataset Overview")

    # ---- Panel 2: Statistical summary heat-table (mean/std/min/max) ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[["mean", "std", "min", "max"]]
        desc_norm = (desc - desc.min()) / (desc.max() - desc.min() + 1e-9)
        sns.heatmap(desc_norm.iloc[:8], annot=desc.iloc[:8].round(1), fmt="",
                    cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontsize": 8, "fontfamily": "Times New Roman"})
    _bold_axis_labels(ax, title="2. Statistical Summary")

    # ---- Panel 3: Missing value analysis ----
    ax = axes[panel]; panel += 1
    miss = df.isnull().mean().sort_values(ascending=False) * 100
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=14))
        ax.axis("off")
    else:
        miss[miss > 0].head(10).plot(kind="bar", ax=ax, color="#c0392b")
    _bold_axis_labels(ax, "Feature", "% Missing", "3. Missing Value Analysis")

    # ---- Panel 4: Class distribution / target distribution ----
    ax = axes[panel]; panel += 1
    if target_col is not None:
        if is_classification_target:
            df[target_col].value_counts().plot(kind="bar", ax=ax, color="#2980b9")
            _bold_axis_labels(ax, "Class", "Count", "4. Class Distribution")
        else:
            sns.histplot(df[target_col], kde=True, ax=ax, color="#2980b9")
            _bold_axis_labels(ax, target_col, "Frequency", "4. Target Distribution")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "No target column supplied", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=12))
        _bold_axis_labels(ax, title="4. Target Distribution")

    # ---- Panel 5: Correlation matrix (heatmap) ----
    ax = axes[panel]; panel += 1
    corr_cols = numeric_cols[:10] if len(numeric_cols) > 10 else numeric_cols
    if len(corr_cols) >= 2:
        sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, ax=ax,
                    cbar=False, annot=len(corr_cols) <= 6, fmt=".2f",
                    annot_kws={"fontsize": 7})
    _bold_axis_labels(ax, title="5. Correlation Matrix")

    # ---- Panel 6: Feature distribution (histogram of 1st numeric feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.histplot(df[feat_a], kde=True, ax=ax, color="#27ae60")
    _bold_axis_labels(ax, feat_a if feat_a else "", "Frequency",
                       "6. Feature Distribution")

    # ---- Panel 7: Box plot (outlier detection) across numeric features ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        plot_cols = numeric_cols[:6]
        df_scaled = (df[plot_cols] - df[plot_cols].mean()) / (df[plot_cols].std() + 1e-9)
        sns.boxplot(data=df_scaled, ax=ax, color="#f39c12")
        ax.tick_params(axis="x", rotation=45)
    _bold_axis_labels(ax, "Feature", "Standardized Value", "7. Box Plot (Outliers)")

    # ---- Panel 8: Violin plot ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.violinplot(y=df[feat_a], ax=ax, color="#8e44ad")
    _bold_axis_labels(ax, "", feat_a if feat_a else "",
                       "8. Violin Plot")

    # ---- Panel 9: Scatter plot (feature 1 vs feature 2, hued by target) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None and feat_b is not None:
        hue = df[target_col] if (target_col and is_classification_target) else None
        sns.scatterplot(x=df[feat_a], y=df[feat_b],
                         hue=hue, ax=ax, palette="Set2", legend=False, s=18)
    _bold_axis_labels(ax, feat_a if feat_a else "", feat_b if feat_b else "",
                       "9. Scatter Plot")

    # ---- Panel 10: Q-Q plot (normality check on highest-variance feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        stats.probplot(df[feat_a].dropna(), dist="norm", plot=ax)
        ax.get_lines()[0].set_markerfacecolor("#2980b9")
        ax.get_lines()[0].set_markeredgecolor("#2980b9")
        ax.get_lines()[1].set_color("#c0392b")
    _bold_axis_labels(ax, "Theoretical Quantiles", "Sample Quantiles", "10. Q-Q Plot")

    # ---- Panel 11: KDE / density plot overlay of top numeric features ----
    ax = axes[panel]; panel += 1
    for c in kde_cols:
        sns.kdeplot(df[c], ax=ax, label=c, linewidth=1.5)
    if kde_cols:
        ax.legend(prop=fm.FontProperties(family="Times New Roman", size=9))
    _bold_axis_labels(ax, "Value", "Density", "11. KDE / Density Plot")

    # ---- Panel 12: Feature importance / variance plot ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        var = df[numeric_cols].var().sort_values(ascending=False).head(8)
        var.plot(kind="barh", ax=ax, color="#16a085")
        ax.invert_yaxis()
    _bold_axis_labels(ax, "Variance", "Feature", "12. Variance / Importance Plot")

    for ax in axes:
        _bold_axis_labels(ax)  # re-apply tick font in case a plotting call reset it

    saved = _save_eps(fig, save_path)
    if saved:
        print(f"[generate_eda_summary] Figure saved -> {saved} (600 DPI, EPS)")
    return fig


# --------------------------------------------------------------------------
# 3. GENERIC REGRESSION TRAIN/EVAL FUNCTION  (Section 4.2)
# --------------------------------------------------------------------------
def train_evaluate_regression(models: dict, X_train, X_test, y_train, y_test,
                               scale=False, verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of regression
    models on the same train/test split.

    Parameters
    ----------
    models : dict {name: sklearn-estimator}
    X_train, X_test, y_train, y_test : array-like
    scale : bool -> StandardScaler applied when True (fit on train only)
    verbose : bool -> print per-model metrics as they are computed

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by R2 desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                                   verbose=verbose, return_dict=True)
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 4. GENERIC CLASSIFICATION TRAIN/EVAL FUNCTION  (Section 4.3)
# --------------------------------------------------------------------------
def train_evaluate_classification(models: dict, X_train, X_test, y_train, y_test,
                                   scale=False, average="weighted", verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of classification
    models on the same train/test split.

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by Accuracy desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = None
        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)
            except Exception:
                y_proba = None
        metrics = classification_performance_metrics(
            y_test, y_pred, y_proba=y_proba, model_name=name,
            average=average, verbose=verbose, return_dict=True, plot=False
        )
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 5. GENERIC REGRESSION METRICS FUNCTION  (Section 4.4)
# --------------------------------------------------------------------------
def regression_performance_metrics(y_true, y_pred, model_name="Model",
                                    verbose=True, return_dict=False):
    """
    Computes and displays ALL standard regression performance metrics:
    MAE, MSE, RMSE, R2, Adjusted R2 (n only), MAPE.
    """
    from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                  r2_score, mean_absolute_percentage_error)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    if verbose:
        print(f"--- Regression Metrics: {model_name} ---")
        print(f"  MAE  : {mae:.4f}")
        print(f"  MSE  : {mse:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R2   : {r2:.4f}")
        print(f"  MAPE : {mape:.2f}%\n")

    result = {"Model": model_name, "MAE": mae, "MSE": mse,
              "RMSE": rmse, "R2": r2, "MAPE(%)": mape}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


# --------------------------------------------------------------------------
# 6. GENERIC CLASSIFICATION METRICS FUNCTION  (Section 4.5)
# --------------------------------------------------------------------------
def classification_performance_metrics(y_true, y_pred, y_proba=None,
                                        model_name="Model", average="weighted",
                                        verbose=True, return_dict=False,
                                        plot=True, save_path=None):
    """
    Computes and displays ALL standard classification performance metrics:
    Accuracy, Precision, Recall, F1-score, ROC-AUC (binary/multiclass ovr),
    and (optionally) plots the confusion matrix using the mandatory lab
    formatting (Times New Roman, bold 15pt axis labels).
    """
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score, confusion_matrix)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)

    roc_auc = np.nan
    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                roc_auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr",
                                         average=average)
        except Exception:
            roc_auc = np.nan

    if verbose:
        print(f"--- Classification Metrics: {model_name} ---")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC   : N/A")
        print()

    if plot:
        set_plot_style()
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontfamily": "Times New Roman", "fontsize": 13})
        _bold_axis_labels(ax, "Predicted Label", "True Label",
                           f"Confusion Matrix \u2013 {model_name}")
        _save_eps(fig, save_path)

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-score": f1, "ROC-AUC": roc_auc}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_validate, GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                               GradientBoostingClassifier, StackingClassifier)
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix)


warnings.filterwarnings("ignore")
RANDOM_STATE = 42

FIG_DIR = "figures"
RES_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

set_plot_style()


## 1. LOAD DATASET AND ENCODE CLASS LABELS

In [2]:
DATA_PATH = "breast_cancer_data.csv"  # <-- upload this file to the notebook's working directory
raw = pd.read_csv(DATA_PATH)

drop_cols = [c for c in raw.columns if c.lower() == "id" or "unnamed" in c.lower()]
df = raw.drop(columns=drop_cols).copy()
feature_names = [c for c in df.columns if c != "diagnosis"]

# target: 0 = malignant (M), 1 = benign (B)
df["target"] = df["diagnosis"].map({"M": 0, "B": 1})
df["diagnosis"] = df["diagnosis"].map({"M": "Malignant", "B": "Benign"})

print("Dataset shape:", df.shape)
print(df["diagnosis"].value_counts())
print("Missing values:", int(df[feature_names].isnull().sum().sum()))

Dataset shape: (569, 32)
diagnosis
Benign       357
Malignant    212
Name: count, dtype: int64
Missing values: 0


## 2. EDA (reusable function from Experiment 1)

In [3]:
eda_df = df.drop(columns=["target"])
generate_eda_summary(
    eda_df, target_col="diagnosis", dataset_name="Wisconsin Breast Cancer (PCA study)",
    save_path=f"{FIG_DIR}/eda_breast_cancer.eps"
)
plt.close("all")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[generate_eda_summary] Figure saved -> figures/eda_breast_cancer.eps (600 DPI, EPS)


## 3. TRAIN / TEST SPLIT + STANDARDIZATION (shared by both settings)

In [4]:
X = df[feature_names].values
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 4. PCA: CHOOSE COMPONENTS (95% explained variance target) + SCREE PLOT

In [5]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_train_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
VARIANCE_TARGET = 0.95
n_components = int(np.argmax(cum_var >= VARIANCE_TARGET) + 1)
print(f"\nPCA: {n_components} components explain "
      f"{cum_var[n_components-1]*100:.2f}% of variance "
      f"(target: {VARIANCE_TARGET*100:.0f}%)")

pca_summary = pd.DataFrame({
    "Setting": ["With-PCA"],
    "Chosen Components": [n_components],
    "Variance Target": [f"{VARIANCE_TARGET*100:.0f}%"],
    "Explained Variance (%)": [cum_var[n_components-1] * 100],
    "Justification": [f"Smallest number of principal components whose cumulative "
                       f"explained variance reaches the {VARIANCE_TARGET*100:.0f}% target."],
})
pca_summary.to_csv(f"{RES_DIR}/pca_variance_summary.csv", index=False)
print(pca_summary)

# Scree plot
fig, ax1 = plt.subplots(figsize=(9, 5.5))
components = np.arange(1, len(pca_full.explained_variance_ratio_) + 1)
ax1.bar(components, pca_full.explained_variance_ratio_ * 100, color="#2980b9", alpha=0.7,
        label="Individual Explained Variance")
ax2 = ax1.twinx()
ax2.plot(components, cum_var * 100, color="#c0392b", marker="o", markersize=3,
         label="Cumulative Explained Variance")
ax2.axhline(VARIANCE_TARGET * 100, linestyle="--", color="gray", linewidth=1)
ax2.axvline(n_components, linestyle="--", color="gray", linewidth=1)
ax2.set_ylabel("Cumulative Explained Variance (%)",
               fontproperties=fm.FontProperties(family="Times New Roman", weight="bold", size=15))
_bold_axis_labels(ax1, "Principal Component", "Individual Explained Variance (%)",
                   f"PCA Scree Plot ({n_components} components -> {VARIANCE_TARGET*100:.0f}% variance)")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right",
           prop=fm.FontProperties(family="Times New Roman", size=11))
_save_eps(fig, f"{FIG_DIR}/pca_scree_plot.eps")
plt.close(fig)

# Apply PCA with the chosen number of components
pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"Reduced feature space: {X_train_scaled.shape[1]} -> {X_train_pca.shape[1]} dimensions")


def time_fit_predict(model, Xtr, Xte, ytr):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    train_t = time.perf_counter() - t0
    t0 = time.perf_counter()
    y_pred = model.predict(Xte)
    pred_t = time.perf_counter() - t0
    return y_pred, train_t, pred_t


def get_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    return None

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



PCA: 10 components explain 95.27% of variance (target: 95%)
    Setting  Chosen Components Variance Target  Explained Variance (%)  \
0  With-PCA                 10             95%               95.267689   

                                       Justification  
0  Smallest number of principal components whose ...  


Reduced feature space: 30 -> 10 dimensions


## 5. HYPERPARAMETER GRIDS (kept small and single-core-friendly; see report

In [6]:
#    for the rationale behind each range)
# --------------------------------------------------------------------------
MODEL_GRIDS = {
    "SVM": (SVC(probability=True, random_state=RANDOM_STATE),
            {"kernel": ["linear", "rbf"], "C": [0.1, 1, 10], "gamma": ["scale", "auto"]}),
    "Naive Bayes": (GaussianNB(),
                    {"var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]}),
    "KNN": (KNeighborsClassifier(),
            {"n_neighbors": [3, 5, 7, 9, 11], "weights": ["uniform", "distance"],
             "metric": ["euclidean", "manhattan"]}),
    "Logistic Regression": (LogisticRegression(max_iter=2000),
                             {"C": [0.01, 0.1, 1, 10, 100], "penalty": ["l2"]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=RANDOM_STATE),
                       {"max_depth": [3, 5, 7, None], "criterion": ["gini", "entropy"]}),
    "Random Forest": (RandomForestClassifier(random_state=RANDOM_STATE),
                       {"n_estimators": [50, 100], "max_depth": [5, 10, None]}),
    "AdaBoost": (AdaBoostClassifier(random_state=RANDOM_STATE),
                 {"n_estimators": [50, 100, 200], "learning_rate": [0.1, 1.0]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=RANDOM_STATE),
                           {"n_estimators": [50, 100], "learning_rate": [0.1, 1.0],
                            "max_depth": [2, 3]}),
    "XGBoost": (XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=1),
                {"n_estimators": [50, 100], "learning_rate": [0.1, 0.3],
                 "max_depth": [3, 5]}),
}

STACKING_BASE = [
    ("svm", SVC(probability=True, random_state=RANDOM_STATE)),
    ("nb", GaussianNB()),
    ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE)),
]


def tune_model(name, estimator, grid, Xtr, ytr):
    gs = GridSearchCV(estimator, grid, cv=cv_strategy, scoring="accuracy", n_jobs=1)
    gs.fit(Xtr, ytr)
    return gs

## 6. RUN EACH MODEL UNDER BOTH SETTINGS (No-PCA and With-PCA)

In [7]:
settings = {"No-PCA": (X_train_scaled, X_test_scaled), "With-PCA": (X_train_pca, X_test_pca)}

all_results = {}          # (model, setting) -> metrics dict
all_best_params = {}      # (model, setting) -> best params
all_cv_fold_scores = {}   # (model, setting) -> array of 5 fold accuracies
fitted_models = {}        # (model, setting) -> fitted best estimator

for setting_name, (Xtr, Xte) in settings.items():
    print(f"\n{'='*70}\nSETTING: {setting_name}  (features: {Xtr.shape[1]})\n{'='*70}")

    for model_name, (estimator, grid) in MODEL_GRIDS.items():
        print(f"\n[{setting_name}] Tuning {model_name}...", flush=True)
        t0 = time.perf_counter()
        gs = tune_model(model_name, estimator, grid, Xtr, y_train)
        tune_time = time.perf_counter() - t0
        best_model = gs.best_estimator_

        y_pred, train_t, pred_t = time_fit_predict(best_model, Xtr, Xte, y_train)
        proba = get_proba(best_model, Xte)

        metrics = classification_performance_metrics(
            y_test, y_pred, proba, model_name=f"{model_name} ({setting_name})",
            return_dict=True, plot=False
        )
        metrics["Training Time (s)"] = train_t
        metrics["Tuning Time (s)"] = tune_time

        cv_res = cross_validate(best_model, Xtr, y_train, cv=cv_strategy, scoring="accuracy")

        all_results[(model_name, setting_name)] = metrics
        all_best_params[(model_name, setting_name)] = gs.best_params_
        all_cv_fold_scores[(model_name, setting_name)] = cv_res["test_score"]
        fitted_models[(model_name, setting_name)] = best_model

        print(f"  Best params: {gs.best_params_}")
        print(f"  Best CV accuracy: {gs.best_score_:.4f} | Test accuracy: {metrics['Accuracy']:.4f} "
              f"| Tune time: {tune_time:.1f}s")

    # Stacking (not part of MODEL_GRIDS since it's not hyperparameter-gridded
    # the same way; base models + meta-learner choice is evaluated directly)
    print(f"\n[{setting_name}] Training Stacking Ensemble...", flush=True)
    stacking_model = StackingClassifier(
        estimators=[(n, e) for n, e in STACKING_BASE],
        final_estimator=LogisticRegression(max_iter=2000),
        cv=cv_strategy
    )
    t0 = time.perf_counter()
    y_pred, train_t, pred_t = time_fit_predict(stacking_model, Xtr, Xte, y_train)
    tune_time = time.perf_counter() - t0
    proba = get_proba(stacking_model, Xte)
    metrics = classification_performance_metrics(
        y_test, y_pred, proba, model_name=f"Stacking ({setting_name})",
        return_dict=True, plot=False
    )
    metrics["Training Time (s)"] = train_t
    metrics["Tuning Time (s)"] = tune_time
    cv_res = cross_validate(stacking_model, Xtr, y_train, cv=cv_strategy, scoring="accuracy")

    all_results[("Stacking", setting_name)] = metrics
    all_best_params[("Stacking", setting_name)] = "SVM+NB+DT / LogisticRegression (fixed)"
    all_cv_fold_scores[("Stacking", setting_name)] = cv_res["test_score"]
    fitted_models[("Stacking", setting_name)] = stacking_model
    print(f"  CV accuracy: {cv_res['test_score'].mean():.4f} | Test accuracy: {metrics['Accuracy']:.4f}")

MODEL_ORDER = ["SVM", "Naive Bayes", "KNN", "Logistic Regression", "Decision Tree",
               "Random Forest", "AdaBoost", "Gradient Boosting", "XGBoost", "Stacking"]


SETTING: No-PCA  (features: 30)

[No-PCA] Tuning SVM...


--- Classification Metrics: SVM (No-PCA) ---
  Accuracy  : 0.9825
  Precision : 0.9825
  Recall    : 0.9825
  F1-score  : 0.9825
  ROC-AUC   : 0.9937

  Best params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
  Best CV accuracy: 0.9758 | Test accuracy: 0.9825 | Tune time: 0.6s

[No-PCA] Tuning Naive Bayes...


--- Classification Metrics: Naive Bayes (No-PCA) ---
  Accuracy  : 0.9298
  Precision : 0.9298
  Recall    : 0.9298
  F1-score  : 0.9298
  ROC-AUC   : 0.9868

  Best params: {'var_smoothing': 1e-09}
  Best CV accuracy: 0.9341 | Test accuracy: 0.9298 | Tune time: 0.0s

[No-PCA] Tuning KNN...


--- Classification Metrics: KNN (No-PCA) ---
  Accuracy  : 0.9649
  Precision : 0.9652
  Recall    : 0.9649
  F1-score  : 0.9647
  ROC-AUC   : 0.9714

  Best params: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'uniform'}
  Best CV accuracy: 0.9714 | Test accuracy: 0.9649 | Tune time: 0.3s

[No-PCA] Tuning Logistic Regression...


--- Classification Metrics: Logistic Regression (No-PCA) ---
  Accuracy  : 0.9737
  Precision : 0.9737
  Recall    : 0.9737
  F1-score  : 0.9736
  ROC-AUC   : 0.9957

  Best params: {'C': 0.1, 'penalty': 'l2'}
  Best CV accuracy: 0.9824 | Test accuracy: 0.9737 | Tune time: 0.1s

[No-PCA] Tuning Decision Tree...


...

--- Classification Metrics: Gradient Boosting (With-PCA) ---
  Accuracy  : 0.9386
  Precision : 0.9384
  Recall    : 0.9386
  F1-score  : 0.9384
  ROC-AUC   : 0.9884



  Best params: {'learning_rate': 1.0, 'max_depth': 2, 'n_estimators': 100}
  Best CV accuracy: 0.9626 | Test accuracy: 0.9386 | Tune time: 3.7s

[With-PCA] Tuning XGBoost...


--- Classification Metrics: XGBoost (With-PCA) ---
  Accuracy  : 0.9474
  Precision : 0.9474
  Recall    : 0.9474
  F1-score  : 0.9474
  ROC-AUC   : 0.9927

  Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
  Best CV accuracy: 0.9648 | Test accuracy: 0.9474 | Tune time: 1.0s

[With-PCA] Training Stacking Ensemble...


--- Classification Metrics: Stacking (With-PCA) ---
  Accuracy  : 0.9649
  Precision : 0.9649
  Recall    : 0.9649
  F1-score  : 0.9649
  ROC-AUC   : 0.9934



  CV accuracy: 0.9670 | Test accuracy: 0.9649


## 7. HYPERPARAMETER TUNING RESULTS TABLES (per model, No-PCA vs With-PCA)

In [8]:
tuning_rows = []
for model_name in MODEL_ORDER:
    row = {"Model": model_name}
    for setting_name in settings:
        params = all_best_params[(model_name, setting_name)]
        acc = all_results[(model_name, setting_name)]["Accuracy"]
        row[f"Best Params ({setting_name})"] = str(params)
        row[f"Test Accuracy ({setting_name})"] = acc
    tuning_rows.append(row)
tuning_df = pd.DataFrame(tuning_rows).set_index("Model")
tuning_df.to_csv(f"{RES_DIR}/hyperparameter_tuning_all_models.csv")
print("\n=== Hyperparameter Tuning Results (all models, both settings) ===")
print(tuning_df[["Test Accuracy (No-PCA)", "Test Accuracy (With-PCA)"]])


=== Hyperparameter Tuning Results (all models, both settings) ===
                     Test Accuracy (No-PCA)  Test Accuracy (With-PCA)
Model                                                                
SVM                                0.982456                  0.964912
Naive Bayes                        0.929825                  0.921053
KNN                                0.964912                  0.956140
Logistic Regression                0.973684                  0.973684
Decision Tree                      0.947368                  0.938596
Random Forest                      0.956140                  0.921053
AdaBoost                           0.956140                  0.947368
Gradient Boosting                  0.947368                  0.938596
XGBoost                            0.947368                  0.947368
Stacking                           0.964912                  0.964912


## 8. 5-FOLD CROSS-VALIDATION RESULTS TABLE (Table 5)

In [9]:
cv_table_rows = []
for model_name in MODEL_ORDER:
    row = {"Model": model_name}
    for setting_name in settings:
        folds = all_cv_fold_scores[(model_name, setting_name)]
        for i, score in enumerate(folds):
            row[f"Fold {i+1} ({setting_name})"] = score
        row[f"Avg ({setting_name})"] = folds.mean()
        row[f"Std ({setting_name})"] = folds.std()
    cv_table_rows.append(row)
cv_table_df = pd.DataFrame(cv_table_rows).set_index("Model")
cv_table_df.to_csv(f"{RES_DIR}/cv_results_all_models.csv")
print("\n=== Table 5: 5-Fold Cross-Validation Results (No-PCA vs With-PCA) ===")
print(cv_table_df[["Avg (No-PCA)", "Avg (With-PCA)", "Std (No-PCA)", "Std (With-PCA)"]])


=== Table 5: 5-Fold Cross-Validation Results (No-PCA vs With-PCA) ===
                     Avg (No-PCA)  Avg (With-PCA)  Std (No-PCA)  \
Model                                                             
SVM                      0.975824        0.978022      0.010767   
Naive Bayes              0.934066        0.916484      0.028656   
KNN                      0.971429        0.964835      0.013187   
Logistic Regression      0.982418        0.982418      0.005383   
Decision Tree            0.931868        0.929670      0.023466   
Random Forest            0.962637        0.956044      0.017855   
AdaBoost                 0.980220        0.951648      0.018906   
Gradient Boosting        0.962637        0.962637      0.013187   
XGBoost                  0.973626        0.964835      0.014906   
Stacking                 0.964835        0.967033      0.018906   

                     Std (With-PCA)  
Model                                
SVM                        0.012038  
Naive Baye

## 9. PERFORMANCE COMPARISON TABLE (all models, both settings)

In [10]:
perf_rows = []
for model_name in MODEL_ORDER:
    for setting_name in settings:
        m = all_results[(model_name, setting_name)]
        perf_rows.append({
            "Model": model_name, "Setting": setting_name,
            "Accuracy": m["Accuracy"], "Precision": m["Precision"],
            "Recall": m["Recall"], "F1-score": m["F1-score"],
            "ROC-AUC": m.get("ROC-AUC", np.nan),
            "Training Time (s)": m["Training Time (s)"],
        })
perf_df = pd.DataFrame(perf_rows)
perf_df.to_csv(f"{RES_DIR}/performance_comparison_all_models.csv", index=False)
print("\n=== Performance Comparison (all models, both settings) ===")
print(perf_df)


=== Performance Comparison (all models, both settings) ===
                  Model   Setting  Accuracy  Precision    Recall  F1-score  \
0                   SVM    No-PCA  0.982456   0.982456  0.982456  0.982456   
1                   SVM  With-PCA  0.964912   0.965858  0.964912  0.965073   
2           Naive Bayes    No-PCA  0.929825   0.929825  0.929825  0.929825   
3           Naive Bayes  With-PCA  0.921053   0.920798  0.921053  0.920849   
4                   KNN    No-PCA  0.964912   0.965185  0.964912  0.964725   
5                   KNN  With-PCA  0.956140   0.956073  0.956140  0.956027   
6   Logistic Regression    No-PCA  0.973684   0.973711  0.973684  0.973616   
7   Logistic Regression  With-PCA  0.973684   0.973711  0.973684  0.973616   
8         Decision Tree    No-PCA  0.947368   0.947440  0.947368  0.947087   
9         Decision Tree  With-PCA  0.938596   0.939042  0.938596  0.938743   
10        Random Forest    No-PCA  0.956140   0.956073  0.956140  0.956027   
11  

## 10. VISUALIZATION: Accuracy comparison bar chart (No-PCA vs With-PCA)

In [11]:
pivot_acc = perf_df.pivot(index="Model", columns="Setting", values="Accuracy").loc[MODEL_ORDER]
fig, ax = plt.subplots(figsize=(12, 6))
pivot_acc.plot(kind="bar", ax=ax, color=["#2980b9", "#c0392b"])
ax.legend(prop=fm.FontProperties(family="Times New Roman", size=12))
_bold_axis_labels(ax, "Model", "Test Accuracy", "Test Accuracy: No-PCA vs With-PCA")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/accuracy_comparison_all_models.eps")
plt.close(fig)

# F1-score comparison
pivot_f1 = perf_df.pivot(index="Model", columns="Setting", values="F1-score").loc[MODEL_ORDER]
fig, ax = plt.subplots(figsize=(12, 6))
pivot_f1.plot(kind="bar", ax=ax, color=["#16a085", "#8e44ad"])
ax.legend(prop=fm.FontProperties(family="Times New Roman", size=12))
_bold_axis_labels(ax, "Model", "Test F1-score", "Test F1-score: No-PCA vs With-PCA")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/f1_comparison_all_models.eps")
plt.close(fig)

# CV stability (std dev) comparison
pivot_std = cv_table_df[["Std (No-PCA)", "Std (With-PCA)"]].loc[MODEL_ORDER]
fig, ax = plt.subplots(figsize=(12, 6))
pivot_std.plot(kind="bar", ax=ax, color=["#2980b9", "#c0392b"])
ax.legend(["No-PCA", "With-PCA"], prop=fm.FontProperties(family="Times New Roman", size=12))
_bold_axis_labels(ax, "Model", "CV Accuracy Std. Dev.", "Cross-Validation Stability: No-PCA vs With-PCA")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/stability_comparison_all_models.eps")
plt.close(fig)

# Training time comparison (log scale, since XGBoost/ensembles differ a lot from GaussianNB)
pivot_time = perf_df.pivot(index="Model", columns="Setting", values="Training Time (s)").loc[MODEL_ORDER]
fig, ax = plt.subplots(figsize=(12, 6))
pivot_time.plot(kind="bar", ax=ax, color=["#2980b9", "#c0392b"], logy=True)
ax.legend(prop=fm.FontProperties(family="Times New Roman", size=12))
_bold_axis_labels(ax, "Model", "Training Time (s, log scale)", "Training Time: No-PCA vs With-PCA")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/training_time_comparison_all_models.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 11. ROC CURVES FOR SELECTED MODELS (best overall + a linear + an ensemble)

In [12]:
#     under both settings
# --------------------------------------------------------------------------
selected_for_roc = ["Logistic Regression", "Random Forest", "XGBoost"]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, setting_name in zip(axes, settings):
    for model_name in selected_for_roc:
        model = fitted_models[(model_name, setting_name)]
        Xte = settings[setting_name][1]
        proba = get_proba(model, Xte)
        if proba is None:
            continue
        fpr, tpr, _ = roc_curve(y_test, proba[:, 1])
        auc = roc_auc_score(y_test, proba[:, 1])
        ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})", linewidth=1.8)
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.legend(fontsize=9)
    _bold_axis_labels(ax, "False Positive Rate", "True Positive Rate", f"ROC Curves ({setting_name})")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/roc_curves_selected_models.eps")
plt.close(fig)

# Confusion matrices for the best-overall model in each setting
best_no_pca = perf_df[perf_df["Setting"] == "No-PCA"].sort_values("Accuracy", ascending=False).iloc[0]
best_with_pca = perf_df[perf_df["Setting"] == "With-PCA"].sort_values("Accuracy", ascending=False).iloc[0]
print(f"\nBest No-PCA model: {best_no_pca['Model']} (Accuracy={best_no_pca['Accuracy']:.4f})")
print(f"Best With-PCA model: {best_with_pca['Model']} (Accuracy={best_with_pca['Accuracy']:.4f})")

for label, row in [("No-PCA", best_no_pca), ("With-PCA", best_with_pca)]:
    model_name = row["Model"]
    model = fitted_models[(model_name, label)]
    Xte = settings[label][1]
    y_pred = model.predict(Xte)
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                annot_kws={"fontfamily": "Times New Roman", "fontsize": 14})
    _bold_axis_labels(ax, "Predicted Label", "True Label",
                       f"Confusion Matrix - Best {label} Model ({model_name})")
    _save_eps(fig, f"{FIG_DIR}/cm_best_{label.lower().replace('-', '_')}.eps")
    plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



Best No-PCA model: SVM (Accuracy=0.9825)
Best With-PCA model: Logistic Regression (Accuracy=0.9737)


## 12. SUMMARY: IMPROVEMENT FROM PCA (per model)

In [13]:
improvement_rows = []
for model_name in MODEL_ORDER:
    acc_no = all_results[(model_name, "No-PCA")]["Accuracy"]
    acc_with = all_results[(model_name, "With-PCA")]["Accuracy"]
    f1_no = all_results[(model_name, "No-PCA")]["F1-score"]
    f1_with = all_results[(model_name, "With-PCA")]["F1-score"]
    std_no = all_cv_fold_scores[(model_name, "No-PCA")].std()
    std_with = all_cv_fold_scores[(model_name, "With-PCA")].std()
    improvement_rows.append({
        "Model": model_name,
        "Accuracy Change": acc_with - acc_no,
        "F1 Change": f1_with - f1_no,
        "Stability Change (Std Dev)": std_with - std_no,
    })
improvement_df = pd.DataFrame(improvement_rows).set_index("Model")
improvement_df.to_csv(f"{RES_DIR}/pca_improvement_summary.csv")
print("\n=== PCA Improvement Summary (With-PCA minus No-PCA) ===")
print(improvement_df)

print("\nAll figures saved under:", os.path.abspath(FIG_DIR))
print("All result tables saved under:", os.path.abspath(RES_DIR))
print("\nDone.")


=== PCA Improvement Summary (With-PCA minus No-PCA) ===
                     Accuracy Change  F1 Change  Stability Change (Std Dev)
Model                                                                      
SVM                        -0.017544  -0.017383                    0.001271
Naive Bayes                -0.008772  -0.008975                   -0.008274
KNN                        -0.008772  -0.008697                    0.002964
Logistic Regression         0.000000   0.000000                    0.000000
Decision Tree              -0.008772  -0.008344                   -0.004306
Random Forest              -0.035088  -0.034786                    0.000533
AdaBoost                   -0.008772  -0.008407                    0.001475
Gradient Boosting          -0.008772  -0.008649                    0.005973
XGBoost                     0.000000   0.000281                    0.001244
Stacking                    0.000000   0.000188                   -0.005006

All figures saved under: /home